In [1]:
%pip install tensorflow tqdm "numpy<2"


Note: you may need to restart the kernel to use updated packages.


### Training Model

In [6]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm.notebook import tqdm

# 1. Load and Preprocess MNIST Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# 2. Setup Configurations
activations = ['relu', 'sigmoid', 'tanh']
hidden_depths = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

# Outer progress bar for the total number of models
pbar = tqdm(total=len(activations) * len(hidden_depths), desc='Total Models Progress')

os.makedirs('./models', exist_ok=True)

for act in activations:
    for depth in hidden_depths:
        model_name = f'model_{act}_{depth}layer'
        save_path = f'./models/{model_name}.keras'

        # Create the MLP model
        model = models.Sequential()
        model.add(layers.Input(shape=(28, 28, 1)))
        model.add(layers.Flatten())

        for _ in range(depth):
            model.add(layers.Dense(128, activation=act))

        model.add(layers.Dense(10, activation='softmax'))

        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        history = model.fit(
            x_train,
            y_train,
            epochs=8,
            batch_size=64,
            validation_data=(x_test, y_test),
            verbose=1
        )

        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
        print(f'{model_name} -> test_loss: {test_loss:.4f}, test_acc: {test_acc:.4f}')

        model.save(save_path)
        pbar.update(1)

pbar.close()
print('All models successfully trained and saved.')


Total Models Progress:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9154 - loss: 0.3029 - val_accuracy: 0.9543 - val_loss: 0.1558
Epoch 2/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9621 - loss: 0.1345 - val_accuracy: 0.9646 - val_loss: 0.1185
Epoch 3/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9724 - loss: 0.0940 - val_accuracy: 0.9717 - val_loss: 0.0940
Epoch 4/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9788 - loss: 0.0716 - val_accuracy: 0.9749 - val_loss: 0.0822
Epoch 5/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9827 - loss: 0.0565 - val_accuracy: 0.9774 - val_loss: 0.0768
Epoch 6/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9869 - loss: 0.0452 - val_accuracy: 0.9778 - val_loss: 0.0728
Epoch 7/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9892 - loss: 0.0358 - val_accuracy: 0.9785 - val_loss: 0.0717
Epoch 8/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9916 - loss: 0.0297 - val_accuracy: 0.9785 - v

### Testing Model

In [ ]:
import os
import numpy as np
from tensorflow.keras import models

# Paste exactly 28 lines, each with exactly 28 characters.
# Use 0 for background and 1 for ink.
manual_digit_text = """
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
0000000000000000000000000000
""".strip()

rows = [line.strip() for line in manual_digit_text.splitlines() if line.strip()]
if len(rows) != 28:
    raise ValueError(f"manual_digit_text must have 28 rows, got {len(rows)}")
if any(len(row) != 28 for row in rows):
    bad = [len(row) for row in rows]
    raise ValueError(f"each row must have 28 characters, got lengths {bad}")

manual_digit = np.array([[int(ch) for ch in row] for row in rows], dtype=np.float32)
manual_digit = manual_digit.reshape(1, 28, 28, 1)

activations = ["relu", "sigmoid", "tanh"]
hidden_depths = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

for act in activations:
    for depth in hidden_depths:
        model_name = f"model_{act}_{depth}layer"
        model_path = f"./models/{model_name}.keras"

        if os.path.exists(model_path):
            model = models.load_model(model_path)
            pred = model.predict(manual_digit, verbose=0)
            digit = int(np.argmax(pred[0]))
            confidence = float(np.max(pred[0]))
            print(f"{model_name} -> predicted: {digit}, confidence: {confidence:.4f}")
        else:
            print(f"{model_name} not found")


model_relu_1layer -> predicted: 3, confidence: 0.9447
model_relu_2layer -> predicted: 3, confidence: 0.9815
model_relu_3layer -> predicted: 3, confidence: 1.0000
model_relu_4layer -> predicted: 3, confidence: 0.7640
model_relu_5layer -> predicted: 3, confidence: 0.9955
model_relu_6layer -> predicted: 3, confidence: 0.9957
model_relu_7layer -> predicted: 3, confidence: 0.4263
model_relu_8layer -> predicted: 3, confidence: 0.9990
model_relu_9layer -> predicted: 3, confidence: 0.9802
model_relu_10layer -> predicted: 3, confidence: 1.0000
model_sigmoid_1layer -> predicted: 3, confidence: 0.9542
model_sigmoid_2layer -> predicted: 3, confidence: 0.3985
model_sigmoid_3layer -> predicted: 3, confidence: 0.9875
model_sigmoid_4layer -> predicted: 3, confidence: 0.4293
model_sigmoid_5layer -> predicted: 2, confidence: 0.6475
model_sigmoid_6layer -> predicted: 3, confidence: 0.9875
model_sigmoid_7layer -> predicted: 3, confidence: 0.9954
model_sigmoid_8layer -> predicted: 3, confidence: 0.9945
mod